# SAE splicing: causally testing sparse-autoencoder features

Sparse autoencoders (SAEs) decompose a layer's activations into (hopefully)
interpretable features. Training one only shows the features *reconstruct*
the activations -- the standard causal test is **splicing**: swap the SAE's
reconstruction back into the model at the site it was trained on, replay the
downstream computation, and measure how much the model's behavior changes.
If behavior is preserved, the features carry the information the model
actually uses; ablating one feature then measures its individual causal role.

TorchLens packages the whole experiment as one call,
`torchlens.bridge.sae.splice(...)` (documented-unstable spelling), built on
the intervention primitives (`fork` + `push` + `tl.splice_module`). This
notebook runs it end to end on a small model with a synthetic SAE trained
here in the notebook -- no pretrained SAE download needed, and any object
with an `encode`/`decode` pair (e.g. an SAE Lens `SAE`) drops in the same
way.

In [1]:
import torch
from torch import nn

import torchlens as tl
from torchlens.bridge import sae as tl_sae

torch.manual_seed(0)

## 1. A small trained model

A two-layer MLP trained on a toy 8-dimensional classification task, so the
splice has real behavior to preserve or destroy.

In [2]:
class ToyMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.in_proj = nn.Linear(8, 32)
        self.out_proj = nn.Linear(32, 4)

    def forward(self, x):
        return self.out_proj(torch.relu(self.in_proj(x)))


def make_task(n):
    x = torch.randn(n, 8)
    y = (x[:, :4].sum(dim=1) > 0).long() + 2 * (x[:, 4:].sum(dim=1) > 0).long()
    return x, y


model = ToyMLP()
x_train, y_train = make_task(2048)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
for _ in range(300):
    optimizer.zero_grad()
    loss = nn.functional.cross_entropy(model(x_train), y_train)
    loss.backward()
    optimizer.step()
model.eval()

x_test, y_test = make_task(256)
with torch.no_grad():
    clean_accuracy = (model(x_test).argmax(dim=1) == y_test).float().mean().item()
print(f"final train loss {loss.item():.3f}, test accuracy {clean_accuracy:.2%}")

final train loss 0.022, test accuracy 96.88%


## 2. Collect site activations with TorchLens and train a synthetic SAE

One `tl.trace` gives us the ReLU activations the SAE trains on. The SAE is a
standard tied-bias ReLU autoencoder with an L1 sparsity penalty -- tiny, but
structurally the real thing.

In [3]:
site = "relu_1_2"
train_log = tl.trace(model, x_train, save=tl.func("relu"))
activations = train_log[site].out.detach()
print(f"training SAE on {site}: {tuple(activations.shape)}")


class TinySAE(nn.Module):
    def __init__(self, d_in, d_hidden):
        super().__init__()
        self.enc = nn.Linear(d_in, d_hidden)
        self.dec = nn.Linear(d_hidden, d_in)

    def encode(self, x):
        return torch.relu(self.enc(x))

    def decode(self, latents):
        return self.dec(latents)


sae = TinySAE(32, 128)
sae_optimizer = torch.optim.Adam(sae.parameters(), lr=1e-3)
for _ in range(1500):
    sae_optimizer.zero_grad()
    latents = sae.encode(activations)
    reconstruction = sae.decode(latents)
    sae_loss = (reconstruction - activations).pow(2).mean() + 1e-4 * latents.abs().mean()
    sae_loss.backward()
    sae_optimizer.step()
sae.eval()
print(f"final SAE loss {sae_loss.item():.4f}")

training SAE on relu_1_2: (2048, 32)


final SAE loss 0.0011


## 3. The splice experiment

`splice` captures a clean intervention-ready trace on the test inputs, forks
it, replaces the site activation with the SAE reconstruction, replays the
downstream computation on TorchLens's replay engine, and reports both
reconstruction fidelity and the output-level causal effect.

In [4]:
result = tl_sae.splice(model, x_test, site=site, sae=sae)

print(f"spliced site:                  {result.site}")
print(f"reconstruction MSE:            {result.reconstruction_mse:.5f}")
print(f"fraction variance explained:   {result.fraction_variance_explained:.2%}")
print(
    f"output delta (L2 / max abs):   {result.output_delta_l2:.4f} / {result.output_delta_max:.4f}"
)

spliced_accuracy = (result.spliced_outputs[0].argmax(dim=1) == y_test).float().mean().item()
print(f"accuracy clean -> spliced:     {clean_accuracy:.2%} -> {spliced_accuracy:.2%}")

spliced site:                  relu_1_2:1
reconstruction MSE:            0.00105
fraction variance explained:   99.96%
output delta (L2 / max abs):   4.4575 / 0.5457
accuracy clean -> spliced:     96.88% -> 96.88%


The reconstruction explains most of the variance and the spliced model keeps
(nearly) the clean accuracy: the SAE features carry the information the model
actually uses at this site. Both traces stay inspectable --
`result.clean` and `result.spliced` are ordinary TorchLens traces, so every
downstream activation of the spliced run is available for comparison.

## 4. Feature-level causal test

`latents_edit=` transforms the encoded latents before decoding -- the knob
for asking whether an individual feature is causally real. Here we ablate the
single most active latent feature and compare against ablating a random one.

In [5]:
with torch.no_grad():
    mean_latent_activity = sae.encode(result.clean_site_out).mean(dim=0)
top_feature = int(mean_latent_activity.argmax())
random_feature = int(torch.randint(0, mean_latent_activity.numel(), (1,)))


def ablate(feature_index):
    def edit(latents):
        edited = latents.clone()
        edited[..., feature_index] = 0.0
        return edited

    return edit


for name, feature in [("top", top_feature), ("random", random_feature)]:
    ablated = tl_sae.splice(model, x_test, site=site, sae=sae, latents_edit=ablate(feature))
    accuracy = (ablated.spliced_outputs[0].argmax(dim=1) == y_test).float().mean().item()
    print(
        f"ablate {name} feature {feature:>3}: "
        f"output L2 delta {ablated.output_delta_l2:.4f}, accuracy {accuracy:.2%}"
    )

ablate top feature  52: output L2 delta 22.6775, accuracy 97.27%


ablate random feature  89: output L2 delta 6.7527, accuracy 96.88%


Ablating the most active feature moves the outputs far more than ablating a
random (mostly dead) one -- a per-feature causal footprint, measured through
the same verified replay path.

## Where to go next

- `result.spliced.draw()` renders the spliced computation graph with the
  intervention site marked.
- `tl.splice_module` accepts any `nn.Module`, so the same pattern splices
  transcoders or probes; `attach_hooks` + `push` (what `splice` does inside)
  composes multiple interventions on one fork.
- A real pretrained SAE (e.g. from SAE Lens) drops in wherever `TinySAE` is
  used: anything exposing `encode`/`decode` works.